# Experiment: Variables acústicas relevantes

**Pregunta.** ¿Qué características (MFCC, log-mel, ZCR, centroide, bandwidth, rolloff) separan sirena de tráfico y, en menor medida, los tipos etiquetados?

**Criterio de éxito.** Resumen por clase, figuras comparativas y una conclusión operativa: log-mel 2D para CNN, MFCC como baseline compacto.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEED = 7

def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/working"),
        Path("/kaggle/input/doppler-ml"),
    ]
    for parent in Path.cwd().resolve().parents:
        candidates.append(parent)
    for cand in candidates:
        if (cand / "src" / "paths.py").exists():
            return cand
    return Path.cwd()

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import corpus_paths, figures_dir, is_kaggle, tables_dir

print("repo:", REPO_ROOT)
print("kaggle:", is_kaggle())
print("available:", list(corpus_paths().available()))
SEED


repo: /home/jeancdevx/dev/doppler/doppler-ml
kaggle: False
available: ['sirennet', 'lssiren', 'urbansound8k']


7

## Plan

- Hipótesis 1: sirena vs tráfico se separa bien en centroide/rolloff y en los MFCC bajos (tonalidad periódica).
- Hipótesis 2: ambulance/police/firetruck se solapan más que sirena vs traffic.
- Muestra: hasta 40 clips por clase de sireNNet (reproducible con SEED=7).
- Métricas: media ± std de descriptores; mapa log-mel y curva MFCC de un ejemplo.


In [2]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src.audio_features import (
    HAS_LIBROSA,
    load_audio,
    log_mel_spectrogram,
    mfcc,
    spectral_bandwidth,
    spectral_centroid,
    spectral_rolloff,
    zero_crossing_rate,
)

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(SEED)
FIG = figures_dir()
TAB = tables_dir()
print("librosa:", HAS_LIBROSA)

inv_path = TAB / "file_inventory.csv"
inventory = pd.read_csv(inv_path) if inv_path.exists() else pd.DataFrame()
from pathlib import Path as _P
if not inventory.empty:
    inventory = inventory[inventory["path"].map(lambda p: _P(str(p)).exists())]
inventory.groupby(["corpus", "label"]).size() if not inventory.empty else "empty"


librosa: False


corpus    label    
sirennet  ambulance    400
          firetruck    400
          police       454
          traffic      421
dtype: int64

## Muestreo estratificado y extracción


In [3]:
MAX_PER_CLASS = 40
sirennet = inventory[inventory["corpus"] == "sirennet"] if not inventory.empty else pd.DataFrame()
sample_df = pd.DataFrame()

if sirennet.empty:
    print("sireNNet no disponible")
else:
    parts = []
    for label, sub in sirennet.groupby("label"):
        parts.append(sub.sample(min(MAX_PER_CLASS, len(sub)), random_state=SEED))
    sample_df = pd.concat(parts, ignore_index=True)

rows = []
for _, row in sample_df.iterrows():
    clip = load_audio(row["path"], duration=3.0)
    mf = mfcc(clip, n_mfcc=13)
    rows.append({
        "path": row["path"],
        "label": row["label"],
        "binary": "traffic" if row["label"] == "traffic" else "siren",
        "zcr_mean": float(np.mean(zero_crossing_rate(clip))),
        "centroid_mean": float(np.mean(spectral_centroid(clip))),
        "bandwidth_mean": float(np.mean(spectral_bandwidth(clip))),
        "rolloff_mean": float(np.mean(spectral_rolloff(clip))),
        **{f"mfcc_{i+1}_mean": float(np.mean(mf[i])) for i in range(min(13, mf.shape[0]))},
    })

feat = pd.DataFrame(rows)
if not feat.empty:
    feat.to_csv(TAB / "sirennet_feature_sample.csv", index=False)
    desc = feat.groupby("label")[["zcr_mean", "centroid_mean", "bandwidth_mean", "rolloff_mean"]].agg(["mean", "std"])
    desc.to_csv(TAB / "sirennet_descriptor_summary.csv")
feat.head() if not feat.empty else feat


,path,label,binary,zcr_mean,centroid_mean,bandwidth_mean,rolloff_mean,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,mfcc_4_mean,mfcc_5_mean,mfcc_6_mean,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean
0,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.120742,2119.331167,2202.722315,3784.912482,-937.069908,182.524950,-88.394986,-6.151545,22.906682,9.654523,16.347269,33.974206,2.974515,8.614150,12.007022,-3.782923,-10.759718
1,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.109343,1776.179397,1922.344585,2578.559980,-1262.391075,220.780187,-78.313411,-104.180751,-18.000341,8.326323,7.238437,9.382495,-19.429943,-22.258545,11.763489,-8.240251,-12.379008
2,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.163732,2344.692221,2162.647533,4073.227278,-1091.377019,145.327831,-100.110921,23.214988,-0.658287,21.740430,33.302707,37.875946,-13.149082,1.222034,-4.003461,-7.936693,-23.926308
3,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.104175,1443.031488,930.069575,1951.631456,-1768.331547,301.988330,-280.324864,-103.575138,-46.387168,23.082076,17.216112,28.936644,-22.811532,-33.932302,-6.540904,3.197149,-7.480740
4,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,ambulance,siren,0.134347,1823.714960,1105.951214,2776.961325,-1947.454799,173.950696,-214.140642,20.927712,-18.380091,96.431533,105.630034,30.034235,-86.011651,-47.015044,77.679357,95.622689,-55.014366


## Descriptores espectrales por clase


In [4]:
if feat.empty:
    print("Sin características: monta sireNNet y re-ejecuta")
else:
    long = feat.melt(
        id_vars=["label", "binary"],
        value_vars=["zcr_mean", "centroid_mean", "bandwidth_mean", "rolloff_mean"],
        var_name="feature",
        value_name="value",
    )
    g = sns.catplot(data=long, x="label", y="value", col="feature", kind="box", sharey=False, height=3.2, aspect=0.95, color="#3b6d9a")
    g.set_xticklabels(rotation=35)
    g.savefig(FIG / "descriptors_by_class.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_19152/3952155383.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## MFCC medios: sirena vs tráfico y entre tipos


In [5]:
if not feat.empty:
    mfcc_cols = [c for c in feat.columns if c.startswith("mfcc_")]
    means = feat.groupby("label")[mfcc_cols].mean()
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, series in means.iterrows():
        ax.plot(range(1, len(series) + 1), series.values, marker="o", label=label)
    ax.set_xlabel("Coeficiente MFCC")
    ax.set_ylabel("Media en la muestra")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "mfcc_means_by_class.png", bbox_inches="tight")
    plt.show()
    means


/tmp/ipykernel_19152/3616100810.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ejemplo log-mel vs MFCC

La CNN 2D opera de forma natural sobre el mapa log-mel (imagen tiempo-frecuencia). El MFCC comprime ese mapa a pocos coeficientes: útil para baselines clásicos, menos expresivo para redes convolucionales.


In [6]:
if sirennet.empty:
    print("sin ejemplo")
else:
    pick = {}
    for label, sub in sirennet.groupby("label"):
        pick[label] = sub.sample(1, random_state=SEED).iloc[0]["path"]
    labels = list(pick)
    fig, axes = plt.subplots(len(labels), 2, figsize=(10, 2.3 * len(labels)), squeeze=False)
    for i, label in enumerate(labels):
        clip = load_audio(pick[label], duration=3.0)
        mel = log_mel_spectrogram(clip)
        mf = mfcc(clip, n_mfcc=13)
        axes[i, 0].imshow(mel, origin="lower", aspect="auto", cmap="magma")
        axes[i, 0].set_ylabel(label)
        axes[i, 1].imshow(mf, origin="lower", aspect="auto", cmap="coolwarm")
        if i == 0:
            axes[i, 0].set_title("Log-mel")
            axes[i, 1].set_title("MFCC")
    fig.tight_layout()
    fig.savefig(FIG / "logmel_vs_mfcc_examples.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_19152/4207304514.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultados

- **Detección:** ZCR, centroide y rolloff, más los MFCC bajos, son candidatos fuertes para sirena vs tráfico.
- **Tipo:** se espera más solapamiento; el mapa log-mel conserva modulación temporal (wail/yelp/hi-lo) que el vector MFCC promedio pierde.
- **Decisión de representación:** CNN sobre log-mel 2D como modelo principal; MFCC (13–20) como baseline tabular/SVM/CNN 1D.
- Siguiente fase: splits agrupados y entrenamiento, no más EDA.


In [7]:
result = {
    "seed": SEED,
    "librosa": HAS_LIBROSA,
    "n_feature_rows": int(len(feat)) if "feat" in globals() else 0,
    "figures": sorted(p.name for p in FIG.glob("*.png")),
}
result


{'seed': 7,
 'librosa': False,
 'n_feature_rows': 160,
 'figures': ['class_balance.png',
  'descriptors_by_class.png',
  'duration_hist.png',
  'logmel_vs_mfcc_examples.png',
  'mfcc_means_by_class.png',
  'sirennet_waveform_stft.png',
  'urbansound8k_class_counts.png']}